# ACE-Step song generation (Colab T4, 30-second test)

Text-to-music with vocals using [ACE-Step v1-3.5B](https://github.com/ace-step/ACE-Step).

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Run the cells top to bottom. The first run downloads about 8 GB of weights, which takes a few minutes. After that, each 30-second clip takes well under a minute.

Notes for T4:
- T4 has no fast bfloat16 support, so the model runs in **float16**, the same as the official ACE-Step Colab.
- If a clip comes out silent or as noise (float16 overflow), set `DTYPE = "bfloat16"` in step 3, run it again, then run step 5 again. bfloat16 is slower on a T4, but it matches the precision the model was trained in.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Install ACE-Step

ACE-Step's `requirements.txt` pins old versions (e.g. `spacy==3.8.4`) that have no builds for Colab's Python 3.13. So this cell installs ACE-Step with `--no-deps`, then installs only what inference needs, with relaxed versions. The exception is `py3langid`, which stays at 0.3.0 because 0.4.0 removed an API that ACE-Step calls. It keeps Colab's own `torch`, `transformers` and `huggingface_hub`.

The Japanese-lyrics extras (`cutlet`, `fugashi`) are installed on their own line, so if they fail to build, the rest of the install still works.

**Expected noise:** pip will print `ERROR: pip's dependency resolver ... ace-step 0.2.0 requires X==..., but you have X ...`. That's a warning about ACE-Step's own pins, not a failure. `pytorch_lightning` and `tensorboardX` are only used for training. If the version table prints at the end, the install worked.

In [ ]:
!pip install -q --no-deps git+https://github.com/ace-step/ACE-Step.git
!pip install -q "diffusers>=0.33.0" "spacy>=3.8.7,<3.9" loguru pypinyin "py3langid==0.3.0" hangul-romanize num2words soundfile librosa peft accelerate
!pip install -q cutlet "fugashi[unidic-lite]" || echo "Japanese lyric extras failed to install; only needed for Japanese lyrics."
import importlib.metadata as md
for pkg in ["ace-step", "torch", "transformers", "diffusers", "spacy", "py3langid"]:
    print(f"{pkg:13s}", md.version(pkg))

## 2b. Hugging Face login (optional)

The ACE-Step weights are public, so you don't need a token. With one, the 8 GB download is faster and avoids anonymous rate limits.

Add the token as a Colab secret. Click the 🔑 **Secrets** icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on **Notebook access**. Don't paste the token into a cell, because it would be saved in the notebook.

In [ ]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as e:  # secret missing or notebook access not granted
    token = None
    print(f"No HF_TOKEN secret found ({type(e).__name__}); downloading anonymously.")
if token:
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")

## 3. Load the model

`CPU_OFFLOAD` keeps only the model stage that is currently running on the GPU. In float16 everything fits in the T4's 15 GB, so it is off by default. Turn it on if you hit CUDA out-of-memory errors.

In [ ]:
import os, time
import torch
DTYPE = "float16"  #@param ["float16", "bfloat16", "float32"]
CPU_OFFLOAD = False  #@param {type:"boolean"}
CHECKPOINT_DIR = "/content/ace_step_checkpoints"  #@param {type:"string"}

# ACE-Step reads this env var and it overrides the constructor's dtype argument.
os.environ["ACE_PIPELINE_DTYPE"] = DTYPE

import numpy as np
import soundfile as sf
from acestep.pipeline_ace_step import ACEStepPipeline

# Newer torchaudio routes save() through torchcodec, which Colab may not have.
# Write the WAV with soundfile directly instead.
def _save_wav_file(self, target_wav, idx, save_path=None, sample_rate=48000, format="wav"):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    wav = target_wav.float().cpu().numpy().T  # (channels, samples) -> (samples, channels)
    sf.write(save_path, wav, sample_rate)
    return save_path
ACEStepPipeline.save_wav_file = _save_wav_file

t0 = time.time()
pipe = ACEStepPipeline(
    checkpoint_dir=CHECKPOINT_DIR,
    cpu_offload=CPU_OFFLOAD,
    torch_compile=False,
    overlapped_decode=False,
)
pipe.load_checkpoint(pipe.checkpoint_dir)
pipe.loaded = True
print(f"Loaded in {time.time() - t0:.0f}s | dtype={pipe.dtype} | "
      f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 4. Describe the song

- **Prompt:** comma-separated tags for genre, instruments, mood, tempo and vocal style.
- **Lyrics:** use `[verse]`, `[chorus]` and `[bridge]` section tags. For an instrumental, set `LYRICS = "[instrumental]"`.
- 30 seconds covers about one verse and one chorus, so keep the lyrics short.

In [ ]:
PROMPT = "pop, female vocals, acoustic guitar, piano, upbeat, catchy, 110 bpm, bright, warm"  #@param {type:"string"}
DURATION = 30  #@param {type:"slider", min:10, max:60, step:5}
INFER_STEPS = 60  #@param {type:"slider", min:20, max:100, step:5}
GUIDANCE_SCALE = 15.0  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}

LYRICS = """[verse]
Morning light across the floor
Coffee steam and an open door
Every street is calling out my name
Nothing here will ever feel the same

[chorus]
Oh we're running with the summer sun
Hearts on fire, we're just getting started
Oh we're running till the day is done
"""

## 5. Generate

In [ ]:
from IPython.display import Audio, display

os.makedirs("/content/outputs", exist_ok=True)
out_path = f"/content/outputs/acestep_{time.strftime('%Y%m%d_%H%M%S')}_seed{SEED}.wav"

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
result = pipe(
    format="wav",
    audio_duration=float(DURATION),
    prompt=PROMPT,
    lyrics=LYRICS,
    infer_step=INFER_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    scheduler_type="euler",
    cfg_type="apg",
    omega_scale=10.0,
    manual_seeds=[SEED],
    guidance_interval=0.5,
    guidance_interval_decay=0.0,
    min_guidance_scale=3.0,
    use_erg_tag=True,
    use_erg_lyric=True,
    use_erg_diffusion=True,
    oss_steps=None,
    guidance_scale_text=0.0,
    guidance_scale_lyric=0.0,
    save_path=out_path,
    batch_size=1,
)
elapsed = time.time() - t0
wav_path = result[0]

# Sanity check: float16 overflow shows up as NaN/Inf, silence, or full-scale noise.
audio, sr = sf.read(wav_path)
finite = bool(np.isfinite(audio).all())
peak = float(np.nanmax(np.abs(audio))) if audio.size else 0.0
rms = float(np.sqrt(np.nanmean(audio ** 2))) if audio.size else 0.0
print(f"Generated {len(audio) / sr:.1f}s in {elapsed:.0f}s -> {wav_path}")
print(f"peak={peak:.3f}  rms={rms:.4f}  peak VRAM={torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
if not finite or peak < 1e-3 or rms > 0.5:
    print("WARNING: output looks broken (NaN, silence or noise). "
          "Set DTYPE='bfloat16' in step 3, run it again, then run this cell again.")

display(Audio(wav_path))

## 6. Download (optional)

In [ ]:
from google.colab import files
files.download(wav_path)

## 7. Save to Google Drive (optional)

This can also cache the model weights on Drive so the next session skips the 8 GB download. To use the cache, set `CHECKPOINT_DIR = "/content/drive/MyDrive/ace_step_checkpoints"` in step 3.

In [ ]:
from google.colab import drive
import shutil
CACHE_WEIGHTS = False  #@param {type:"boolean"}

drive.mount("/content/drive")
dst = "/content/drive/MyDrive/ace_step_outputs"
os.makedirs(dst, exist_ok=True)
shutil.copy(wav_path, dst)
print("Copied to", dst)

if CACHE_WEIGHTS and not CHECKPOINT_DIR.startswith("/content/drive"):
    shutil.copytree(CHECKPOINT_DIR, "/content/drive/MyDrive/ace_step_checkpoints", dirs_exist_ok=True)
    print("Weights cached to /content/drive/MyDrive/ace_step_checkpoints")

---
# Part 2: lofi versions of your songs

This part uses ACE-Step's **audio2audio** mode. Each of your songs is the starting point, and the model re-renders it in the lofi style from the prompt, singing the lyrics you paste in step 9b. Run steps 1 to 3 first so the model is loaded.

- **30-second test:** the output is as long as the input, so each song is cut to a 30-second excerpt first. By default the excerpt starts 35% into the song, which usually lands in a verse or chorus.
- **No repeats:** each song is rendered once. Duplicates are skipped if the file is identical or the title is the same (compilation prefixes like `[DDR] 100 Love Songs - 01 -` are ignored). Songs already rendered in this session are also skipped, so a rerun only does new songs, or songs whose lyrics you changed.
- **Your rights:** only use songs you own or have permission to use. A lofi edit of someone else's record is still their record. This matches the repo's `lofify --i-own-this` rule.

## 8. Add your songs

Leave `UPLOAD` ticked to pick files from your computer, such as the ones in `my-songs/`. To use a Drive folder instead, untick it and set `SONGS_DIR`, for example `/content/drive/MyDrive/my-songs` (mount Drive in step 7 first).

In [ ]:
import hashlib, re
from pathlib import Path

SONGS_DIR = "/content/my-songs"  #@param {type:"string"}
UPLOAD = True  #@param {type:"boolean"}
I_OWN_THESE = False  #@param {type:"boolean"}
assert I_OWN_THESE, ("Tick I_OWN_THESE to confirm these are songs you own, are licensed "
                     "to use, or are public domain.")

AUDIO_EXTS = {".mp3", ".wav", ".flac", ".ogg", ".m4a"}
songs_dir = Path(SONGS_DIR)
songs_dir.mkdir(parents=True, exist_ok=True)
if UPLOAD:
    from google.colab import files
    for name, data in files.upload().items():
        (songs_dir / name).write_bytes(data)

def song_title(path):
    """'[DDR] 100 Love Songs - 01 - Guzaarish' -> 'Guzaarish'."""
    stem = re.sub(r"[\[(].*?[\])]", " ", Path(path).stem)
    parts = [p.strip() for p in stem.split(" - ") if p.strip() and not p.strip().isdigit()]
    return parts[-1] if parts else Path(path).stem

def title_key(path):
    return re.sub(r"[^a-z0-9]+", "", song_title(path).lower())

songs, by_hash, by_title = [], {}, {}
for p in sorted(songs_dir.rglob("*")):
    if p.suffix.lower() not in AUDIO_EXTS:
        continue
    digest = hashlib.sha1(p.read_bytes()).hexdigest()
    key = title_key(p)
    if digest in by_hash:
        print(f"skip  {p.name}  (same file as {by_hash[digest]})")
    elif key in by_title:
        print(f"skip  {p.name}  (same song as {by_title[key]})")
    else:
        by_hash[digest] = by_title[key] = p.name
        songs.append((p, digest))

print(f"\n{len(songs)} unique song(s):")
for p, _ in songs:
    print(" -", song_title(p), f"({p.name})")

## 9. Lofi settings

- **`STRENGTH`** sets how much of the original survives. Higher values keep more of the melody, structure and vocal timing; lower values give the model more freedom, so it sounds more lofi but less like your song. With vocals, 0.5 to 0.7 usually keeps the sung lines close to the original.
- Each song gets its own seed (`BASE_SEED` plus its position in the list), so the songs don't all come out with the same groove.
- The prompt describes the lofi backing. Each song's vocal style is added from step 9b.

In [ ]:
LOFI_PROMPT = "lofi hip hop, chill, mellow, dusty boom bap drums, soft rhodes piano, warm bass, jazzy chords, relaxed, 75 bpm"  #@param {type:"string"}
STRENGTH = 0.5  #@param {type:"slider", min:0.2, max:0.9, step:0.05}
EXCERPT_SECONDS = 30  #@param {type:"slider", min:10, max:60, step:5}
START_FRACTION = 0.35  #@param {type:"slider", min:0.0, max:0.9, step:0.05}
LOFI_STEPS = 60  #@param {type:"slider", min:20, max:100, step:5}
LOFI_GUIDANCE = 15.0  #@param {type:"number"}
BASE_SEED = 1000  #@param {type:"integer"}

## 9b. Lyrics for each song

To keep the vocals, ACE-Step needs the words sung in each excerpt. Paste them into `lyrics` for each song.

- **Only paste the lines inside the excerpt**, meaning the 30 seconds starting at `start`. Lines from elsewhere in the song won't line up with the original melody. Set `start` (in seconds) to a part you know the words for, such as the first chorus. Leave it as `None` to use `START_FRACTION`.
- **Romanized Hindi works best**, e.g. `Tujhe dekha to` rather than Devanagari. ACE-Step's Hindi text cleanup is an unfinished placeholder, while romanized lines go through its well-tested English path.
- Add section tags like `[verse]` or `[chorus]` above the lines.
- `vocals` is added to the prompt for that song. The entries below are guesses, so check them against the originals and fix them if they're wrong.
- A song with empty `lyrics` is skipped, so nothing comes out instrumental by accident. To make one instrumental on purpose, set its lyrics to `"[instrumental]"`.

In [ ]:
SONG_LYRICS = {
    "Guzaarish": {
        "start": None,
        "vocals": "soft male vocals, breathy, intimate",
        "lyrics": """
""",
    },
    "Ishq Vishq": {
        "start": None,
        "vocals": "male and female duet vocals, soft, warm",
        "lyrics": """
""",
    },
    "Dil To Bachcha Hai": {
        "start": None,
        "vocals": "soulful male vocals, gentle, warm",
        "lyrics": """
""",
    },
}

lyrics_by_key = {title_key(t): v for t, v in SONG_LYRICS.items()}
for p, _ in songs:
    entry = lyrics_by_key.get(title_key(p))
    if entry is None:
        print(f"{song_title(p)}: not in SONG_LYRICS yet. Add an entry for it.")
    elif not entry["lyrics"].strip():
        print(f"{song_title(p)}: no lyrics yet, will be skipped")
    else:
        n = sum(1 for l in entry["lyrics"].splitlines() if l.strip() and not l.strip().startswith("["))
        print(f"{song_title(p)}: {n} sung line(s), start={entry['start']}")

## 10. Render the lofi versions

In [ ]:
import json
import librosa
from IPython.display import Audio, display
from acestep.music_dcae.music_dcae_pipeline import MusicDCAE

# ACE-Step reads the reference with torchaudio.load, which needs torchcodec on
# newer torchaudio.  Read it with soundfile instead (excerpts are WAV).
def _load_audio(self, audio_path):
    audio, sr = sf.read(audio_path, dtype="float32", always_2d=True)
    audio = torch.from_numpy(audio.T.copy())
    if audio.shape[0] == 1:
        audio = audio.repeat(2, 1)
    return audio, sr
MusicDCAE.load_audio = _load_audio

out_dir = Path("/content/lofi_outputs")
excerpt_dir = out_dir / "excerpts"
excerpt_dir.mkdir(parents=True, exist_ok=True)
manifest_path = out_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

for i, (src, digest) in enumerate(songs):
    title = song_title(src)
    safe = re.sub(r"[^\w\-]+", "_", title).strip("_") or f"song_{i}"
    entry = lyrics_by_key.get(title_key(src))
    if not entry or not entry["lyrics"].strip():
        print(f"skip  {title}  (no lyrics in step 9b)")
        continue
    lyrics = entry["lyrics"].strip()
    lyrics_id = hashlib.sha1(lyrics.encode("utf-8")).hexdigest()[:10]
    done = manifest.get(digest)
    if done and done.get("lyrics_id") == lyrics_id and Path(done["output"]).exists():
        print(f"skip  {title}  (already rendered with these lyrics: {done['output']})")
        continue

    total = librosa.get_duration(path=str(src))
    wanted = entry["start"] if entry["start"] is not None else total * START_FRACTION
    start = max(0.0, min(total - EXCERPT_SECONDS, float(wanted)))
    prompt = f"{LOFI_PROMPT}, {entry['vocals']}" if entry.get("vocals") else LOFI_PROMPT
    y, _ = librosa.load(str(src), sr=44100, mono=False, offset=start, duration=EXCERPT_SECONDS)
    if y.ndim == 1:
        y = np.stack([y, y])
    excerpt = excerpt_dir / f"{safe}.wav"
    sf.write(excerpt, y.T, 44100)

    seed = BASE_SEED + i
    out = out_dir / f"{safe}_lofi_s{int(STRENGTH * 100)}_seed{seed}.wav"
    print(f"\n[{i + 1}/{len(songs)}] {title}: {start:.0f}s to {start + EXCERPT_SECONDS:.0f}s "
          f"of {total:.0f}s, strength={STRENGTH}, seed={seed}")
    t0 = time.time()
    pipe(
        format="wav",
        audio_duration=float(EXCERPT_SECONDS),
        prompt=prompt,
        lyrics=lyrics,
        infer_step=LOFI_STEPS,
        guidance_scale=LOFI_GUIDANCE,
        scheduler_type="euler",
        cfg_type="apg",
        omega_scale=10.0,
        manual_seeds=[seed],
        guidance_interval=0.5,
        guidance_interval_decay=0.0,
        min_guidance_scale=3.0,
        use_erg_tag=True,
        use_erg_lyric=True,
        use_erg_diffusion=True,
        oss_steps=None,
        guidance_scale_text=0.0,
        guidance_scale_lyric=0.0,
        audio2audio_enable=True,
        ref_audio_strength=STRENGTH,
        ref_audio_input=str(excerpt),
        task="audio2audio",
        save_path=str(out),
        batch_size=1,
    )

    audio, sr = sf.read(out)
    peak = float(np.nanmax(np.abs(audio))) if audio.size else 0.0
    rms = float(np.sqrt(np.nanmean(audio ** 2))) if audio.size else 0.0
    print(f"done in {time.time() - t0:.0f}s -> {out.name}  (peak={peak:.3f}, rms={rms:.4f})")
    if not np.isfinite(audio).all() or peak < 1e-3 or rms > 0.5:
        print("WARNING: output looks broken (NaN, silence or noise); "
              "try DTYPE='bfloat16' in step 3.")

    print("original excerpt:"); display(Audio(str(excerpt)))
    print("lofi version:");     display(Audio(str(out)))

    manifest[digest] = {"title": title, "source": src.name, "output": str(out),
                        "excerpt_start": round(start, 1), "strength": STRENGTH,
                        "seed": seed, "prompt": prompt, "lyrics_id": lyrics_id}
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

print(f"\nAll done. {len(manifest)} song(s) in {out_dir}")

## 11. Download all lofi versions (optional)

This zips the lofi WAVs and `manifest.json`, which records the settings used for each song. The excerpts are left out.

In [ ]:
import zipfile
from google.colab import files
zip_path = "/content/lofi_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(out_dir.glob("*_lofi_*.wav")) + [manifest_path]:
        zf.write(f, f.name)
files.download(zip_path)